# Preprocessing

In [3]:
import multiprocessing as mp
import os
import sys
from functools import partial
from itertools import chain

import numpy as np
import pandas as pd
import trimesh
from numba import njit
from scipy.spatial import KDTree
from tqdm import tqdm

# make multiprocessing compatible with macOS
# mp.set_start_method('fork', force=True)

In [4]:
# Go up THREE levels (project root directory)
project_root = os.path.dirname(os.path.dirname(os.getcwd()))
# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")

Project root added to sys.path


In [ ]:
from utils import gro_processing as gp

# Data directory can be accessed due to root PATH we set previously
path = 'data/npt-HK4.gro'
file = os.path.join(project_root, path)

# Extracts data from .gro file into multi-index DataFrame (unsorted)
df_gro, title, num_atoms, box_dimensions = gp.read_gro(file, multiply=10, positions=True, velocities=False) # convert nm to Å

# Dictionary of molecules {res_id: [(atom_name, np.array([x, y, z])), ...]}
molecules = {}

for res_id in df_gro.index.get_level_values('res_id').unique():
    residue_data = df_gro.xs(res_id, level='res_id')
    # itertuples() - much faster for large DataFrames
    atom_list = [(row.Index, np.array([row.x, row.y, row.z])) for row in residue_data.itertuples()]
    molecules[res_id] = atom_list
    
    
# Checking
# df_gro
# molecules # display 

# Generating Molecule

In [ ]:
from utils.generate_mol_meshes import molecules_to_meshes

mol_meshes = molecules_to_meshes(molecules, box_dimensions, num_processes=4)
print(len(mol_meshes))        # 1501
print(mol_meshes[1])          # trimesh.Trimesh object